# Etapa 5 — Rede Bayesiana e inferência verificável

Rede com seis nós e três modalidades em `regime_em`. UF, lag e a variável da Etapa 4 são compartilhados com as etapas anteriores. Todos os parâmetros são estimados só com o treino.

**Fonte:** BancoVDE 2026.xlsx / MJSP. **Semente:** 42.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_root():
    """Localiza a raiz do projeto (pasta com scripts/utils.py e data/) a partir do diretório atual."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "scripts" / "utils.py").exists() and (base / "data").exists():
            return base
    for pattern in ("*/scripts/utils.py", "*/*/scripts/utils.py"):   # ex.: Colab em /content
        for hit in sorted(here.glob(pattern)):
            if (hit.parent.parent / "data").exists():
                return hit.parent.parent
    raise FileNotFoundError(
        "Raiz do projeto não encontrada: abra o notebook com o diretório atual "
        "dentro da pasta FIA_P3_SegurancaPublica_Equipe5."
    )

ROOT = _find_root()

sys.path.insert(0, str(ROOT / "scripts"))
sys.modules.pop("utils", None)

from utils import (
    panel,
    training_split,
    prepare_splits,
    temporal_split,
    write_metrics
)

np.random.seed(42)

(ROOT / "outputs").mkdir(exist_ok=True)

print("Configuração da Etapa 5 concluída!")

Configuração da Etapa 5 concluída!


In [2]:
import itertools

from em import hurdle_em, hurdle_posterior

from bayes import (
    PARENTS,
    NODES,
    fit_cpt,
    query,
    param_counts,
    to_pgmpy
)

p, tr, te, _ = training_split()

em = hurdle_em(
    tr.lag1,
    variance_floor=0.05,
    tol=1e-5
)

tr = tr.copy()
te = te.copy()

tr["regime_em"] = np.argmax(
    hurdle_posterior(tr.lag1, em),
    axis=1
).astype(int)

te["regime_em"] = np.argmax(
    hurdle_posterior(te.lag1, em),
    axis=1
).astype(int)

tr["crescimento"] = (
    tr.contagem > tr.lag1
).astype(int)

te["crescimento"] = (
    te.contagem > te.lag1
).astype(int)

mdl = fit_cpt(tr, alpha=1.0)

joint_p, bn_p = param_counts(mdl)

print("DAG:", PARENTS)
print("Parâmetros conjunta:", joint_p)
print("Parâmetros BN:", bn_p)
print("Economia:", joint_p - bn_p)

DAG: {'uf': [], 'semestre2': [], 'lag_alto': ['uf'], 'regime_em': ['uf'], 'alto': ['uf', 'lag_alto', 'regime_em'], 'crescimento': ['semestre2', 'alto']}
Parâmetros conjunta: 1295
Parâmetros BN: 274
Economia: 1021


## Estrutura causal hipotética e d-separação
As arestas codificam uma **fatoração associativa**, sem estabelecer causalidade. `uf→alto`, `lag_alto→alto` e `regime_em→alto` representam dependência da classe em localização e histórico; `uf→lag_alto` e `uf→regime_em` capturam variação de escala e regimes; `alto→crescimento←semestre2` forma o colisor de diagnóstico. Pelo critério gráfico, `semestre2` está d-separado de `alto` e de `uf` quando nada é observado. Condicionar no colisor `crescimento` abre um caminho estatístico.

In [3]:
ev = {
    "uf": "AM",
    "lag_alto": 1,
    "regime_em": 2,
    "semestre2": 1
}

p_h = query("alto", 1, ev, mdl)
p_g = query("crescimento", 1, ev, mdl)

print("P(alto | e):", p_h)
print("P(crescimento | e):", p_g)

base = query("semestre2", 1, {}, mdl)

p_unobs = query(
    "semestre2",
    1,
    {"alto": 1},
    mdl
)

p_c0 = query(
    "semestre2",
    1,
    {"alto": 1, "crescimento": 0},
    mdl
)

p_c1 = query(
    "semestre2",
    1,
    {"alto": 1, "crescimento": 1},
    mdl
)

print("\nD-separação:")
print("P(S = 1):", base)
print("P(S = 1 | alto = 1):", p_unobs)

print("\nColisor após observar crescimento:")
print("P(S = 1 | alto = 1, crescimento = 0):", p_c0)
print("P(S = 1 | alto = 1, crescimento = 1):", p_c1)

assert abs(base - p_unobs) < 1e-8

P(alto | e): 0.06250000000000001
P(crescimento | e): 0.37849419831223635

D-separação:
P(S = 1): 0.3698167791706848
P(S = 1 | alto = 1): 0.3698167791706844

Colisor após observar crescimento:
P(S = 1 | alto = 1, crescimento = 0): 0.3036379652388512
P(S = 1 | alto = 1, crescimento = 1): 0.42168543447691204


## Comparação com `pgmpy.VariableElimination`

A inferência própria é comparada com a biblioteca em dez consultas. Os resultados da revisão independente recebidos da equipe ficam documentados com proveniência específica em `outputs/validacao_externa_revisores.json`. Quando `pgmpy` está instalado, as dez consultas são **reexecutadas nesta sessão** e o JSON registra a evidência local em vez da externa.

In [4]:
import itertools
import json

queries = []

for uf, lag, reg, sem in itertools.product(
    ["AM", "SP"],
    [0, 1],
    [0, 1, 2],
    [0, 1]
):
    queries.append({
        "uf": uf,
        "lag_alto": lag,
        "regime_em": reg,
        "semestre2": sem
    })

queries = queries[:10]

q_own = [
    query("alto", 1, ev, mdl)
    for ev in queries
]

try:
    graph, infer = to_pgmpy(mdl)
except ImportError as exc:
    raise ImportError(
        "pgmpy não encontrado. Execute: python -m pip install -r requirements.txt"
    ) from exc

errors = []

for ev, own in zip(queries, q_own):
    q = infer.query(["alto"], evidence=ev, show_progress=False)
    errors.append(abs(own - float(q.get_value(alto=1))))

maxdiff = max(errors)

assert maxdiff < 1e-8, (
    maxdiff,
    "enumeração própria e pgmpy não coincidem"
)

n_queries = len(queries)
status = "executado_e_verificado_localmente"
provenance = "pgmpy.VariableElimination executado na sessão atual do notebook"

print("pgmpy:", n_queries, "consultas; diferença máxima =", maxdiff)

metrics = {
    "n_nodes": len(NODES),
    "states": {
        n: len(x)
        for n, x in mdl["states"].items()
    },
    "joint_params": joint_p,
    "bayesian_params": bn_p,
    "saved_params": joint_p - bn_p,
    "example_high": p_h,
    "example_growth": p_g,
    "p_sem_unobserved": base,
    "p_sem_given_high": p_unobs,
    "p_sem_given_high_c0": p_c0,
    "p_sem_given_high_c1": p_c1,
    "pgmpy_status": status,
    "pgmpy_queries": n_queries,
    "pgmpy_max_abs_diff": maxdiff,
    "pgmpy_proveniencia": provenance
}

write_metrics("etapa5.json", metrics)

.venv/lib/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


.venv/lib/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


pgmpy: 10 consultas; diferença máxima = 5.551115123125783e-17


PosixPath('FIA_P3_SegurancaPublica_Equipe5/outputs/etapa5.json')

## Reprodutibilidade
A rede retoma a UF e a classe de alto volume da etapa simbólica e o regime posterior de EM, efetivamente calculado a partir do treino. O preenchimento das CPT não consulta as linhas do conjunto de teste. Com seis variáveis, a enumeração é computacionalmente viável.

## Segunda d-separação solicitada
Além de `semestre2 ⟂ alto`, a estrutura implica `semestre2 ⟂ uf` sem observação de seus descendentes. A segunda identidade é verificada por enumeração na mesma CPT.

In [5]:
import json

p_sem_given_am = query(
    "semestre2",
    1,
    {"uf": "AM"},
    mdl
)

print("P(S = 1):", base)
print("P(S = 1 | UF = AM):", p_sem_given_am)

assert abs(base - p_sem_given_am) < 1e-10

metrics = json.loads(
    (ROOT / "outputs" / "etapa5.json").read_text()
)

metrics["p_sem_given_uf_AM"] = p_sem_given_am

write_metrics("etapa5.json", metrics)

P(S = 1): 0.3698167791706848
P(S = 1 | UF = AM): 0.36981677917068456


PosixPath('FIA_P3_SegurancaPublica_Equipe5/outputs/etapa5.json')